In [0]:
# ITERATION 1, Cell 1 — One-time setup + Day 1 data

spark.sql("CREATE CATALOG IF NOT EXISTS practice")
spark.sql("CREATE SCHEMA IF NOT EXISTS practice.medallion")
spark.sql("CREATE VOLUME  IF NOT EXISTS practice.medallion.landing")

import os

day1_data = """customer_id,name,city,order_amount,order_date,status
1,Ravi Kumar,Mumbai,1500.00,2024-01-10,completed
2,Priya Shah,Pune,2300.50,2024-01-11,completed
3,Ankit Joshi,Delhi,800.00,2024-01-11,pending
4,Sneha Patil,Nagpur,4200.00,2024-01-12,completed
5,Rahul Verma,Chennai,950.00,2024-01-12,pending
"""

vol = "/Volumes/practice/medallion/landing/"
os.makedirs(vol, exist_ok=True)

with open(f"{vol}orders_day1.csv", "w") as f:
    f.write(day1_data)

print("Day 1 file written!")
display(dbutils.fs.ls(vol))

Day 1 file written!


path,name,size,modificationTime
dbfs:/Volumes/practice/medallion/landing/orders_2024.csv,orders_2024.csv,434,1776692230000
dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,orders_day1.csv,293,1776692461000


In [0]:
# ITERATION 1, Cell 2 — BRONZE LAYER
from pyspark.sql.functions import current_timestamp, lit, col

vol = "/Volumes/practice/medallion/landing/"

df_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{vol}orders_day1.csv") \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file",         col("_metadata.file_path")) \
    .withColumn("batch_id",            lit("day1"))

df_bronze.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("practice.medallion.bronze_orders")

print(f"Rows in Bronze: {spark.table('practice.medallion.bronze_orders').count()}")

Rows in Bronze: 5


In [0]:
%sql
SELECT * FROM  practice.medallion.bronze_orders 

customer_id,name,city,order_amount,order_date,status,ingestion_timestamp,source_file,batch_id
1,Ravi Kumar,Mumbai,1500.0,2024-01-10,completed,2026-04-20T14:05:05.227Z,dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,day1
2,Priya Shah,Pune,2300.5,2024-01-11,completed,2026-04-20T14:05:05.227Z,dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,day1
3,Ankit Joshi,Delhi,800.0,2024-01-11,pending,2026-04-20T14:05:05.227Z,dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,day1
4,Sneha Patil,Nagpur,4200.0,2024-01-12,completed,2026-04-20T14:05:05.227Z,dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,day1
5,Rahul Verma,Chennai,950.0,2024-01-12,pending,2026-04-20T14:05:05.227Z,dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,day1


In [0]:
%sql
select * from practice.medallion.bronze_orders

customer_id,name,city,order_amount,order_date,status,ingestion_timestamp,source_file,batch_id
1,Ravi Kumar,Mumbai,1500.0,2024-01-10,completed,2026-04-20T14:05:05.227Z,dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,day1
2,Priya Shah,Pune,2300.5,2024-01-11,completed,2026-04-20T14:05:05.227Z,dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,day1
3,Ankit Joshi,Delhi,800.0,2024-01-11,pending,2026-04-20T14:05:05.227Z,dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,day1
4,Sneha Patil,Nagpur,4200.0,2024-01-12,completed,2026-04-20T14:05:05.227Z,dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,day1
5,Rahul Verma,Chennai,950.0,2024-01-12,pending,2026-04-20T14:05:05.227Z,dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,day1


In [0]:
%sql
DESCRIBE HISTORY practice.medallion.bronze_orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-04-20T14:05:06.000Z,70900424402277,roshan.65q@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1916525182281872),baea5ffb-dfdf-4b03-b0d0-bed5620c577a,0420-133341-apddjewo-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 3007)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
3,2026-04-20T14:04:59.000Z,70900424402277,roshan.65q@gmail.com,DELETE,"Map(predicate -> [""true""])",null,List(1916525182281872),08398f8e-0c43-44e2-ba81-bd58fc32245f,0420-133341-apddjewo-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 3007, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 6, numDeletionVectorsUpdated -> 0, numDeletedRows -> 5, scanTimeMs -> 6, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
2,2026-04-20T14:04:53.000Z,70900424402277,roshan.65q@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1916525182281872),f4fd2d7b-f5c0-43d8-95cd-6beb4cf9eb83,0420-133341-apddjewo-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 3007)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-04-20T14:04:19.000Z,70900424402277,roshan.65q@gmail.com,DELETE,"Map(predicate -> [""true""])",null,List(1916525182281872),1ccad557-b1ea-48bd-97df-fca5a7c3fa05,0420-133341-apddjewo-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 3004, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 51, numDeletionVectorsUpdated -> 0, numDeletedRows -> 5, scanTimeMs -> 24, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-04-20T14:03:32.000Z,70900424402277,roshan.65q@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1916525182281872),14ce52de-080d-4c5b-9ef7-f047e3261646,0420-133341-apddjewo-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 3004)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


In [0]:
# ITERATION 1, Cell 3 — SILVER FACT TABLE
# Concept: Clean data, enforce types, remove bad rows, NO business aggregation yet.

from pyspark.sql.functions import col, trim, upper, lower, to_date, round as spark_round

df_silver = spark.sql("""
    SELECT
        CAST(customer_id AS INT)        AS customer_id,
        TRIM(name)                      AS customer_name,
        UPPER(TRIM(city))               AS city,
        CAST(order_amount AS DOUBLE)    AS order_amount,
        TO_DATE(order_date, 'yyyy-MM-dd') AS order_date,
        LOWER(TRIM(status))             AS status,
        ingestion_timestamp,
        batch_id
    FROM practice.medallion.bronze_orders
    WHERE customer_id IS NOT NULL
      AND order_amount > 0
      AND status IN ('completed','pending','cancelled')
""")

# *** DEDUPLICATION ***
# If same customer_id + order_date appears twice (reprocessed file), keep latest
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window = Window.partitionBy("customer_id","order_date","order_amount") \
               .orderBy(desc("ingestion_timestamp"))

df_silver_deduped = df_silver \
    .withColumn("rn", row_number().over(window)) \
    .filter("rn = 1") \
    .drop("rn")

df_silver_deduped.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("practice.medallion.silver_order_facts")

print(f"Silver rows: {spark.table('practice.medallion.silver_order_facts').count()}")
display(spark.table("practice.medallion.silver_order_facts"))

Silver rows: 5


customer_id,customer_name,city,order_amount,order_date,status,ingestion_timestamp,batch_id
1,Ravi Kumar,MUMBAI,1500.0,2024-01-10,completed,2026-04-20T14:05:05.227Z,day1
2,Priya Shah,PUNE,2300.5,2024-01-11,completed,2026-04-20T14:05:05.227Z,day1
3,Ankit Joshi,DELHI,800.0,2024-01-11,pending,2026-04-20T14:05:05.227Z,day1
4,Sneha Patil,NAGPUR,4200.0,2024-01-12,completed,2026-04-20T14:05:05.227Z,day1
5,Rahul Verma,CHENNAI,950.0,2024-01-12,pending,2026-04-20T14:05:05.227Z,day1


In [0]:
# ITERATION 1, Cell 4 — SCD TYPE 2 DIMENSION TABLE
# Concept: Track history of customer changes (city, name).
# When a value changes → mark old row expired, insert new row.
# is_current=True means "this is the latest version of this customer"

from pyspark.sql.functions import current_timestamp, lit
from delta.tables import DeltaTable

# Build latest snapshot of each customer from silver facts
df_new_customers = spark.sql("""
    SELECT DISTINCT
        customer_id,
        customer_name,
        city
    FROM practice.medallion.silver_order_facts
""") \
    .withColumn("effective_start_date", current_timestamp()) \
    .withColumn("effective_end_date",   lit(None).cast("timestamp")) \
    .withColumn("is_current",           lit(True))

# First run → just create the table
try:
    DeltaTable.forName(spark, "practice.medallion.silver_dim_customers")
    print("Table exists — running MERGE (SCD Type 2)...")

    dim = DeltaTable.forName(spark, "practice.medallion.silver_dim_customers")

    # Step 1: Expire old rows where city or name changed
    dim.alias("old").merge(
        df_new_customers.alias("new"),
        "old.customer_id = new.customer_id AND old.is_current = true"
    ).whenMatchedUpdate(
        condition="old.city != new.city OR old.customer_name != new.customer_name",
        set={
            "is_current":         "false",
            "effective_end_date": "new.effective_start_date"
        }
    ).execute()

    # Step 2: Insert new/changed rows
    df_to_insert = df_new_customers.alias("new") \
        .join(
            spark.table("practice.medallion.silver_dim_customers")
                 .filter("is_current = true")
                 .alias("existing"),
            on="customer_id",
            how="left_anti"   # only rows NOT already current
        )
    
    # Also insert rows where city/name changed (they got expired above)
    df_changed = spark.sql("""
        SELECT n.customer_id, n.customer_name, n.city,
               n.effective_start_date, n.effective_end_date, n.is_current
        FROM   (SELECT DISTINCT customer_id, customer_name, city,
                       current_timestamp() AS effective_start_date,
                       CAST(NULL AS TIMESTAMP) AS effective_end_date,
                       true AS is_current
                FROM practice.medallion.silver_order_facts) n
        LEFT ANTI JOIN (
               SELECT customer_id, customer_name, city
               FROM practice.medallion.silver_dim_customers
               WHERE is_current = true) e
        ON n.customer_id = e.customer_id
           AND n.city = e.city
    """)

    df_changed.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("practice.medallion.silver_dim_customers")

    print("SCD Type 2 merge complete!")

except Exception as e:
    print(f"First run — creating dimension table. ({e})")
    df_new_customers.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("practice.medallion.silver_dim_customers")

display(spark.table("practice.medallion.silver_dim_customers"))

Table exists — running MERGE (SCD Type 2)...
SCD Type 2 merge complete!


customer_id,customer_name,city,effective_start_date,effective_end_date,is_current
4,Sneha Patil,NAGPUR,2026-04-20T14:22:44.502Z,null,true
1,Ravi Kumar,MUMBAI,2026-04-20T14:22:44.502Z,null,true
5,Rahul Verma,CHENNAI,2026-04-20T14:22:44.502Z,null,true
2,Priya Shah,PUNE,2026-04-20T14:22:44.502Z,null,true
3,Ankit Joshi,DELHI,2026-04-20T14:22:44.502Z,null,true


In [0]:
# ITERATION 1, Cell 5 — GOLD LAYER
# Concept: Business-ready aggregates. Always latest values (SCD Type 1 = just overwrite).
# Downstream BI tools / dashboards read from here.

spark.sql("""
    CREATE OR REPLACE TABLE practice.medallion.gold_revenue_by_city AS
    SELECT
        city,
        COUNT(DISTINCT customer_id)                                     AS unique_customers,
        COUNT(*)                                                        AS total_orders,
        ROUND(SUM(order_amount), 2)                                     AS total_revenue,
        ROUND(AVG(order_amount), 2)                                     AS avg_order_value,
        ROUND(SUM(CASE WHEN status='completed' THEN order_amount END),2) AS completed_revenue,
        MAX(order_date)                                                 AS last_order_date
    FROM practice.medallion.silver_order_facts
    GROUP BY city
    ORDER BY total_revenue DESC
""")

spark.sql("""
    CREATE OR REPLACE TABLE practice.medallion.gold_customer_summary AS
    SELECT
        f.customer_id,
        d.customer_name,
        d.city,
        COUNT(*)                        AS total_orders,
        ROUND(SUM(f.order_amount), 2)   AS lifetime_value,
        MAX(f.order_date)               AS last_order_date,
        SUM(CASE WHEN f.status = 'completed' THEN 1 ELSE 0 END) AS completed_orders,
        ROUND(
          100.0 * SUM(CASE WHEN f.status='completed' THEN 1 ELSE 0 END) / COUNT(*),
        1)                              AS completion_pct
    FROM practice.medallion.silver_order_facts f
    JOIN practice.medallion.silver_dim_customers d
      ON f.customer_id = d.customer_id AND d.is_current = true
    GROUP BY f.customer_id, d.customer_name, d.city
""")

print("=== Gold: Revenue by City ===")
display(spark.table("practice.medallion.gold_revenue_by_city"))
print("=== Gold: Customer Summary ===")
display(spark.table("practice.medallion.gold_customer_summary"))

=== Gold: Revenue by City ===


city,unique_customers,total_orders,total_revenue,avg_order_value,completed_revenue,last_order_date
NAGPUR,1,1,4200.0,4200.0,4200.0,2024-01-12
PUNE,1,1,2300.5,2300.5,2300.5,2024-01-11
MUMBAI,1,1,1500.0,1500.0,1500.0,2024-01-10
CHENNAI,1,1,950.0,950.0,null,2024-01-12
DELHI,1,1,800.0,800.0,null,2024-01-11


=== Gold: Customer Summary ===


customer_id,customer_name,city,total_orders,lifetime_value,last_order_date,completed_orders,completion_pct
4,Sneha Patil,NAGPUR,1,4200.0,2024-01-12,1,100.0
1,Ravi Kumar,MUMBAI,1,1500.0,2024-01-10,1,100.0
5,Rahul Verma,CHENNAI,1,950.0,2024-01-12,0,0.0
2,Priya Shah,PUNE,1,2300.5,2024-01-11,1,100.0
3,Ankit Joshi,DELHI,1,800.0,2024-01-11,0,0.0


In [0]:
# ITERATION 2, Cell 7 — New data arrives
# KEY CHANGES:
#   - Ravi Kumar (id=1) moved from Mumbai → Pune  ← tests SCD Type 2
#   - 3 brand new orders
#   - One duplicate of Day 1 order (tests deduplication)

day2_data = """customer_id,name,city,order_amount,order_date,status
1,Ravi Kumar,Pune,2500.00,2024-01-15,completed
2,Priya Shah,Pune,1800.00,2024-01-15,completed
6,Meera Nair,Bangalore,3100.00,2024-01-15,completed
7,Arjun Das,Hyderabad,750.00,2024-01-16,pending
1,Ravi Kumar,Mumbai,1500.00,2024-01-10,completed
"""
# ^^^ Last row is an EXACT DUPLICATE of day1 — our dedup should handle it

vol = "/Volumes/practice/medallion/landing/"
with open(f"{vol}orders_day2.csv", "w") as f:
    f.write(day2_data)

print("Day 2 file ready!")

Day 2 file ready!


In [0]:
# ITERATION 2, Cell 8 — BRONZE appends new file
# Concept: Bronze never touches old data. Just append. Full history preserved.

from pyspark.sql.functions import current_timestamp, input_file_name, lit

vol = "/Volumes/practice/medallion/landing/"

df_bronze_day2 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{vol}orders_day2.csv") \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file",         col("_metadata.file_path")) \
    .withColumn("batch_id",            lit("day2"))

df_bronze_day2.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("practice.medallion.bronze_orders")

total = spark.table("practice.medallion.bronze_orders").count()
print(f"Total Bronze rows (Day 1 + Day 2): {total}")

# Notice: you can see BOTH batches
display(spark.sql("""
    SELECT batch_id, COUNT(*) as row_count 
    FROM practice.medallion.bronze_orders 
    GROUP BY batch_id
"""))

Total Bronze rows (Day 1 + Day 2): 10


batch_id,row_count
day2,5
day1,5


In [0]:
%sql
-- update practice.medallion.bronze_orders set name="roshan" where customer_id="3"
select *from  practice.medallion.bronze_orders 

customer_id,name,city,order_amount,order_date,status,ingestion_timestamp,source_file,batch_id
3,roshan,Delhi,800.0,2024-01-11,pending,2026-04-20T14:05:05.227Z,dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,day1
1,Ravi Kumar,Mumbai,1500.0,2024-01-10,completed,2026-04-20T14:05:05.227Z,dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,day1
2,Priya Shah,Pune,2300.5,2024-01-11,completed,2026-04-20T14:05:05.227Z,dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,day1
4,Sneha Patil,Nagpur,4200.0,2024-01-12,completed,2026-04-20T14:05:05.227Z,dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,day1
5,Rahul Verma,Chennai,950.0,2024-01-12,pending,2026-04-20T14:05:05.227Z,dbfs:/Volumes/practice/medallion/landing/orders_day1.csv,day1
1,Ravi Kumar,Pune,2500.0,2024-01-15,completed,2026-04-20T14:30:45.933Z,dbfs:/Volumes/practice/medallion/landing/orders_day2.csv,day2
2,Priya Shah,Pune,1800.0,2024-01-15,completed,2026-04-20T14:30:45.933Z,dbfs:/Volumes/practice/medallion/landing/orders_day2.csv,day2
6,Meera Nair,Bangalore,3100.0,2024-01-15,completed,2026-04-20T14:30:45.933Z,dbfs:/Volumes/practice/medallion/landing/orders_day2.csv,day2
7,Arjun Das,Hyderabad,750.0,2024-01-16,pending,2026-04-20T14:30:45.933Z,dbfs:/Volumes/practice/medallion/landing/orders_day2.csv,day2
1,Ravi Kumar,Mumbai,1500.0,2024-01-10,completed,2026-04-20T14:30:45.933Z,dbfs:/Volumes/practice/medallion/landing/orders_day2.csv,day2


In [0]:
# ITERATION 2, Cell 9 — SILVER reprocessed
# The duplicate from Day 2 (Ravi's Day 1 order re-sent) will be caught by dedup

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

df_silver_all = spark.sql("""
    SELECT
        CAST(customer_id AS INT)           AS customer_id,
        TRIM(name)                         AS customer_name,
        UPPER(TRIM(city))                  AS city,
        CAST(order_amount AS DOUBLE)       AS order_amount,
        TO_DATE(order_date, 'yyyy-MM-dd')  AS order_date,
        LOWER(TRIM(status))                AS status,
        ingestion_timestamp,
        batch_id
    FROM practice.medallion.bronze_orders
    WHERE customer_id IS NOT NULL
      AND order_amount > 0
""")

window = Window.partitionBy("customer_id","order_date","order_amount") \
               .orderBy(desc("ingestion_timestamp"))

df_silver_deduped = df_silver_all \
    .withColumn("rn", row_number().over(window)) \
    .filter("rn = 1") \
    .drop("rn")

df_silver_deduped.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("practice.medallion.silver_order_facts")

print(f"Silver rows after dedup: {df_silver_deduped.count()}")
print("(Should be 9, not 10 — duplicate removed)")
display(spark.table("practice.medallion.silver_order_facts").orderBy("customer_id","order_date"))

Silver rows after dedup: 9
(Should be 9, not 10 — duplicate removed)


customer_id,customer_name,city,order_amount,order_date,status,ingestion_timestamp,batch_id
1,Ravi Kumar,MUMBAI,1500.0,2024-01-10,completed,2026-04-20T14:30:45.933Z,day2
1,Ravi Kumar,PUNE,2500.0,2024-01-15,completed,2026-04-20T14:30:45.933Z,day2
2,Priya Shah,PUNE,2300.5,2024-01-11,completed,2026-04-20T14:05:05.227Z,day1
2,Priya Shah,PUNE,1800.0,2024-01-15,completed,2026-04-20T14:30:45.933Z,day2
3,roshan,DELHI,800.0,2024-01-11,pending,2026-04-20T14:05:05.227Z,day1
4,Sneha Patil,NAGPUR,4200.0,2024-01-12,completed,2026-04-20T14:05:05.227Z,day1
5,Rahul Verma,CHENNAI,950.0,2024-01-12,pending,2026-04-20T14:05:05.227Z,day1
6,Meera Nair,BANGALORE,3100.0,2024-01-15,completed,2026-04-20T14:30:45.933Z,day2
7,Arjun Das,HYDERABAD,750.0,2024-01-16,pending,2026-04-20T14:30:45.933Z,day2


In [0]:
%sql
update  practice.medallion.silver_order_facts set city ="test" where customer_id = 3

num_affected_rows
1


In [0]:
# ITERATION 2, Cell 10 — SCD TYPE 2 picks up Ravi's city change
# After this: Ravi should have TWO rows — Mumbai (expired) + Pune (current)

from pyspark.sql.functions import current_timestamp, lit
from delta.tables import DeltaTable

df_new_snapshot = spark.sql("""
    SELECT DISTINCT customer_id, customer_name, city
    FROM practice.medallion.silver_order_facts
""") \
    .withColumn("effective_start_date", current_timestamp()) \
    .withColumn("effective_end_date",   lit(None).cast("timestamp")) \
    .withColumn("is_current",           lit(True))

dim = DeltaTable.forName(spark, "practice.medallion.silver_dim_customers")

# Expire rows where city changed
dim.alias("old").merge(
    df_new_snapshot.alias("new"),
    "old.customer_id = new.customer_id AND old.is_current = true"
).whenMatchedUpdate(
    condition="old.city != new.city OR old.customer_name != new.customer_name",
    set={
        "is_current":         "false",
        "effective_end_date": "new.effective_start_date"
    }
).execute()

# Insert new/changed rows
df_to_insert = spark.sql("""
    SELECT n.customer_id, n.customer_name, n.city,
           current_timestamp() AS effective_start_date,
           CAST(NULL AS TIMESTAMP) AS effective_end_date,
           true AS is_current
    FROM (SELECT DISTINCT customer_id, customer_name, city
          FROM practice.medallion.silver_order_facts) n
    LEFT ANTI JOIN (
          SELECT customer_id, city
          FROM practice.medallion.silver_dim_customers
          WHERE is_current = true) e
    ON n.customer_id = e.customer_id AND n.city = e.city
""")

df_to_insert.write.format("delta").mode("append") \
    .saveAsTable("practice.medallion.silver_dim_customers")

print("=== Customer Dimension after Day 2 ===")
print("Ravi Kumar (id=1) should now have 2 rows!")
display(spark.sql("""
    SELECT customer_id, customer_name, city, is_current,
           effective_start_date, effective_end_date
    FROM practice.medallion.silver_dim_customers
    ORDER BY customer_id, effective_start_date
"""))

=== Customer Dimension after Day 2 ===
Ravi Kumar (id=1) should now have 2 rows!


customer_id,customer_name,city,is_current,effective_start_date,effective_end_date
1,Ravi Kumar,MUMBAI,false,2026-04-20T14:22:44.502Z,2026-04-20T14:35:19.778Z
1,Ravi Kumar,MUMBAI,false,2026-04-20T14:35:27.341Z,2026-04-20T14:39:31.920Z
1,Ravi Kumar,PUNE,false,2026-04-20T14:35:27.341Z,2026-04-20T14:39:31.920Z
1,Ravi Kumar,MUMBAI,false,2026-04-20T14:39:38.266Z,2026-04-20T14:48:24.805Z
1,Ravi Kumar,PUNE,false,2026-04-20T14:39:38.266Z,2026-04-20T14:48:24.805Z
1,Ravi Kumar,MUMBAI,true,2026-04-20T14:48:30.312Z,null
1,Ravi Kumar,PUNE,true,2026-04-20T14:48:30.312Z,null
2,Priya Shah,PUNE,true,2026-04-20T14:22:44.502Z,null
3,Ankit Joshi,DELHI,false,2026-04-20T14:22:44.502Z,2026-04-20T14:39:31.920Z
3,roshan,DELHI,false,2026-04-20T14:39:38.266Z,2026-04-20T14:48:24.805Z


In [0]:
# ITERATION 2, Cell 11 — Refresh Gold tables

spark.sql("""
    CREATE OR REPLACE TABLE practice.medallion.gold_revenue_by_city AS
    SELECT
        city,
        COUNT(DISTINCT customer_id)                                      AS unique_customers,
        COUNT(*)                                                         AS total_orders,
        ROUND(SUM(order_amount), 2)                                      AS total_revenue,
        ROUND(AVG(order_amount), 2)                                      AS avg_order_value,
        ROUND(SUM(CASE WHEN status='completed' THEN order_amount END),2) AS completed_revenue,
        MAX(order_date)                                                  AS last_order_date
    FROM practice.medallion.silver_order_facts
    GROUP BY city ORDER BY total_revenue DESC
""")

spark.sql("""
    CREATE OR REPLACE TABLE practice.medallion.gold_customer_summary AS
    SELECT
        f.customer_id,
        d.customer_name,
        d.city,
        COUNT(*)                      AS total_orders,
        ROUND(SUM(f.order_amount), 2) AS lifetime_value,
        MAX(f.order_date)             AS last_order_date,
        SUM(CASE WHEN f.status='completed' THEN 1 ELSE 0 END) AS completed_orders
    FROM practice.medallion.silver_order_facts f
    JOIN practice.medallion.silver_dim_customers d
      ON f.customer_id = d.customer_id AND d.is_current = true
    GROUP BY f.customer_id, d.customer_name, d.city
""")

print("=== Gold: Revenue by City (Day 2) ===")
display(spark.table("practice.medallion.gold_revenue_by_city"))
print("=== Gold: Customer Summary (Ravi now shows Pune) ===")
display(spark.table("practice.medallion.gold_customer_summary"))

=== Gold: Revenue by City (Day 2) ===


city,unique_customers,total_orders,total_revenue,avg_order_value,completed_revenue,last_order_date
PUNE,2,3,6600.5,2200.17,6600.5,2024-01-15
NAGPUR,1,1,4200.0,4200.0,4200.0,2024-01-12
BANGALORE,1,1,3100.0,3100.0,3100.0,2024-01-15
MUMBAI,1,1,1500.0,1500.0,1500.0,2024-01-10
CHENNAI,1,1,950.0,950.0,null,2024-01-12
test,1,1,800.0,800.0,null,2024-01-11
HYDERABAD,1,1,750.0,750.0,null,2024-01-16


=== Gold: Customer Summary (Ravi now shows Pune) ===


customer_id,customer_name,city,total_orders,lifetime_value,last_order_date,completed_orders
4,Sneha Patil,NAGPUR,1,4200.0,2024-01-12,1
5,Rahul Verma,CHENNAI,1,950.0,2024-01-12,0
6,Meera Nair,BANGALORE,1,3100.0,2024-01-15,1
7,Arjun Das,HYDERABAD,1,750.0,2024-01-16,0
2,Priya Shah,PUNE,2,4100.5,2024-01-15,2
3,roshan,test,1,800.0,2024-01-11,0
1,Ravi Kumar,MUMBAI,2,4000.0,2024-01-15,2
1,Ravi Kumar,PUNE,2,4000.0,2024-01-15,2


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DataFrameExample").getOrCreate()

# DataFrame 1
data = [
    (101, "C001", 25, 15000, "2023-01-15", "2025-06-30"),
    (102, "C002", 32, 25000, "2022-11-20", "2025-06-30"),
    (103, "C003", 45, 5000,  "2024-02-10", "2025-06-30"),
    (104, "C004", 29, 18000, "2023-08-05", "2025-06-30"),
    (105, "C005", 38, 30000, "2021-12-01", "2025-06-30"),
    (106, "C006", 50, 12000, "2022-05-14", "2025-06-30"),
    (107, "C007", 27, 8000,  "2024-01-22", "2025-06-30"),
    (108, "C008", 41, 45000, "2021-09-30", "2025-06-30"),
    (109, "C009", 35, 22000, "2023-04-18", "2025-06-30"),
    (110, "C010", 30, 16000, "2022-07-25", "2025-06-30")
]

columns = [
    "ProductID",
    "CustID",
    "CustAge",
    "Out_Bal",
    "Ori_Date",
    "Write_I_Of_Date"
]

df = spark.createDataFrame(data, columns)

df.show(truncate=False)
df.printSchema()

+---------+------+-------+-------+----------+---------------+
|ProductID|CustID|CustAge|Out_Bal|Ori_Date  |Write_I_Of_Date|
+---------+------+-------+-------+----------+---------------+
|101      |C001  |25     |15000  |2023-01-15|2025-06-30     |
|102      |C002  |32     |25000  |2022-11-20|2025-06-30     |
|103      |C003  |45     |5000   |2024-02-10|2025-06-30     |
|104      |C004  |29     |18000  |2023-08-05|2025-06-30     |
|105      |C005  |38     |30000  |2021-12-01|2025-06-30     |
|106      |C006  |50     |12000  |2022-05-14|2025-06-30     |
|107      |C007  |27     |8000   |2024-01-22|2025-06-30     |
|108      |C008  |41     |45000  |2021-09-30|2025-06-30     |
|109      |C009  |35     |22000  |2023-04-18|2025-06-30     |
|110      |C010  |30     |16000  |2022-07-25|2025-06-30     |
+---------+------+-------+-------+----------+---------------+

root
 |-- ProductID: long (nullable = true)
 |-- CustID: string (nullable = true)
 |-- CustAge: long (nullable = true)
 |-- Out_Bal

In [0]:
from pyspark.sql.functions import to_date, col

df = df.withColumn("Ori_Date", to_date(col("Ori_Date"), "yyyy-MM-dd"))

In [0]:
df.show()

+---------+------+-------+-------+----------+---------------+
|ProductID|CustID|CustAge|Out_Bal|  Ori_Date|Write_I_Of_Date|
+---------+------+-------+-------+----------+---------------+
|      101|  C001|     25|  15000|2023-01-15|     2025-06-30|
|      102|  C002|     32|  25000|2022-11-20|     2025-06-30|
|      103|  C003|     45|   5000|2024-02-10|     2025-06-30|
|      104|  C004|     29|  18000|2023-08-05|     2025-06-30|
|      105|  C005|     38|  30000|2021-12-01|     2025-06-30|
|      106|  C006|     50|  12000|2022-05-14|     2025-06-30|
|      107|  C007|     27|   8000|2024-01-22|     2025-06-30|
|      108|  C008|     41|  45000|2021-09-30|     2025-06-30|
|      109|  C009|     35|  22000|2023-04-18|     2025-06-30|
|      110|  C010|     30|  16000|2022-07-25|     2025-06-30|
+---------+------+-------+-------+----------+---------------+



In [0]:
product_data = [
    (101, "Home Loan"),
    (102, "Personal Loan"),
    (103, "Car Loan"),
    (104, "Credit Card"),
    (105, "Education Loan"),
    (106, "Gold Loan"),
    (107, "Business Loan"),
    (108, "Mortgage Loan"),
    (109, "Consumer Loan"),
    (110, "Two Wheeler Loan")
]

product_columns = [
    "Prod_ID",
    "Prod_Name"
]

product_df = spark.createDataFrame(product_data, product_columns)

product_df.show(truncate=False)
product_df.printSchema()

+-------+----------------+
|Prod_ID|Prod_Name       |
+-------+----------------+
|101    |Home Loan       |
|102    |Personal Loan   |
|103    |Car Loan        |
|104    |Credit Card     |
|105    |Education Loan  |
|106    |Gold Loan       |
|107    |Business Loan   |
|108    |Mortgage Loan   |
|109    |Consumer Loan   |
|110    |Two Wheeler Loan|
+-------+----------------+

root
 |-- Prod_ID: long (nullable = true)
 |-- Prod_Name: string (nullable = true)

